In [1]:

!pip install -q transformers datasets peft accelerate \
    scikit-learn evaluate matplotlib pandas


In [2]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        ),
        "GB"
    )
else:
    print("No GPU detected. Please enable GPU runtime.")


PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU Memory: 14.56 GB


In [3]:
# Load Banking77
from datasets import load_dataset

# Download the official Banking77 dataset
dataset = load_dataset("PolyAI/banking77")

# Check available dataset splits
print(dataset)

# Display one example
print("\nFirst training example:")
print(dataset["train"][0])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/13.0k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})

First training example:
{'text': 'I am still waiting on my card?', 'label': 11}


In [4]:
# Understand the available labels
# Extract all category names
label_names = dataset["train"].features["label"].names

print("Total number of categories:", len(label_names))

print("\nAvailable categories:")
for index, name in enumerate(label_names):
    print(index, ":", name)


Total number of categories: 77

Available categories:
0 : activate_my_card
1 : age_limit
2 : apple_pay_or_google_pay
3 : atm_support
4 : automatic_top_up
5 : balance_not_updated_after_bank_transfer
6 : balance_not_updated_after_cheque_or_cash_deposit
7 : beneficiary_not_allowed
8 : cancel_transfer
9 : card_about_to_expire
10 : card_acceptance
11 : card_arrival
12 : card_delivery_estimate
13 : card_linking
14 : card_not_working
15 : card_payment_fee_charged
16 : card_payment_not_recognised
17 : card_payment_wrong_exchange_rate
18 : card_swallowed
19 : cash_withdrawal_charge
20 : cash_withdrawal_not_recognised
21 : change_pin
22 : compromised_card
23 : contactless_not_working
24 : country_support
25 : declined_card_payment
26 : declined_cash_withdrawal
27 : declined_transfer
28 : direct_debit_payment_not_recognised
29 : disposable_card_limits
30 : edit_personal_details
31 : exchange_charge
32 : exchange_rate
33 : exchange_via_app
34 : extra_charge_on_statement
35 : failed_transfer
36 : f

In [5]:
# Check category distribution
import pandas as pd

# Convert training data into a DataFrame
train_df = dataset["train"].to_pandas()

# Convert numeric labels into category names
train_df["intent"] = train_df["label"].apply(
    lambda x: label_names[x]
)

# Count examples in each category
category_counts = (
    train_df["intent"]
    .value_counts()
    .sort_values(ascending=False)
)

print(category_counts.to_string())


intent
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
transaction_charged_twice                           175
declined_cash_withdrawal                            173
transfer_fee_charged                                172
balance_not_updated_after_bank_transfer             171
transfer_not_received_by_recipient                  171
request_refund                                      169
card_payment_not_recognised                         168
card_payment_wrong_exchange_rate                    167
extra_charge_on_statement                           166
wrong_exchange_rate_for_cash_withdrawal             163
Refund_not_showing_up                               162
reverted_card_payment?                              161
cash_withdrawal_not_recognised           

In [6]:
# Filter the dataset

# Our six selected banking intents
selected_intents = [
    "cash_withdrawal_charge",
    "cash_withdrawal_not_recognised",
    "declined_cash_withdrawal",
    "pending_cash_withdrawal",
    "failed_transfer",
    "card_swallowed"
]

# Convert category names to their numeric IDs
selected_label_ids = [
    label_names.index(intent)
    for intent in selected_intents
]

# Keep only the selected categories
train_filtered = dataset["train"].filter(
    lambda example: example["label"] in selected_label_ids
)

test_filtered = dataset["test"].filter(
    lambda example: example["label"] in selected_label_ids
)

print("Training examples:", len(train_filtered))
print("Testing examples:", len(test_filtered))

print("\nSelected categories:")
for intent in selected_intents:
    print("-", intent)


Filter:   0%|          | 0/10003 [00:00<?, ? examples/s]

Filter:   0%|          | 0/3080 [00:00<?, ? examples/s]

Training examples: 851
Testing examples: 240

Selected categories:
- cash_withdrawal_charge
- cash_withdrawal_not_recognised
- declined_cash_withdrawal
- pending_cash_withdrawal
- failed_transfer
- card_swallowed


In [7]:
# Create the splits

# First, split training data into train and validation
split_dataset = train_filtered.train_test_split(
    test_size=0.10,
    seed=42,
    stratify_by_column="label"
)

train_data = split_dataset["train"]
validation_data = split_dataset["test"]

# Keep official test data completely separate
test_data = test_filtered

print("Training samples:", len(train_data))
print("Validation samples:", len(validation_data))
print("Testing samples:", len(test_data))


Training samples: 765
Validation samples: 86
Testing samples: 240


In [8]:
# Check category distribution in each split

def show_distribution(data, split_name):
    df = data.to_pandas()
    df["intent"] = df["label"].apply(
        lambda x: label_names[x]
    )

    print(f"\n{split_name} distribution:")
    print(
        df["intent"]
        .value_counts()
        .reindex(selected_intents, fill_value=0)
    )

show_distribution(train_data, "TRAIN")
show_distribution(validation_data, "VALIDATION")
show_distribution(test_data, "TEST")



TRAIN distribution:
intent
cash_withdrawal_charge            159
cash_withdrawal_not_recognised    144
declined_cash_withdrawal          155
pending_cash_withdrawal           129
failed_transfer                   123
card_swallowed                     55
Name: count, dtype: int64

VALIDATION distribution:
intent
cash_withdrawal_charge            18
cash_withdrawal_not_recognised    16
declined_cash_withdrawal          18
pending_cash_withdrawal           14
failed_transfer                   14
card_swallowed                     6
Name: count, dtype: int64

TEST distribution:
intent
cash_withdrawal_charge            40
cash_withdrawal_not_recognised    40
declined_cash_withdrawal          40
pending_cash_withdrawal           40
failed_transfer                   40
card_swallowed                    40
Name: count, dtype: int64


In [9]:
# Dataset validation

assert len(train_data) > 0, "Training data is empty"
assert len(validation_data) > 0, "Validation data is empty"
assert len(test_data) >= 50, "Need at least 50 test examples"

for intent in selected_intents:
    label_id = label_names.index(intent)

    assert any(
        x == label_id for x in train_data["label"]
    ), f"Missing training examples: {intent}"

    assert any(
        x == label_id for x in test_data["label"]
    ), f"Missing test examples: {intent}"

print("All dataset validation checks passed!")


All dataset validation checks passed!


In [10]:
# Save our prepared dataset
import os

os.makedirs("data", exist_ok=True)

def save_split(data, filename):
    df = data.to_pandas()
    df["intent"] = df["label"].apply(
        lambda x: label_names[x]
    )
    df.to_csv(
        f"data/{filename}.csv",
        index=False
    )

save_split(train_data, "train")
save_split(validation_data, "validation")
save_split(test_data, "test")

print("Dataset files saved successfully!")


Dataset files saved successfully!


In [11]:
# Step 3 - Load Pretrained Model

from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully!")

# Test tokenization
sample_text = "My ATM withdrawal failed."

tokens = tokenizer.tokenize(sample_text)
token_ids = tokenizer.encode(sample_text)

print("\nOriginal text:", sample_text)
print("Tokens:", tokens)
print("Token IDs:", token_ids)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded successfully!

Original text: My ATM withdrawal failed.
Tokens: ['My', 'ĠATM', 'Ġwithdrawal', 'Ġfailed', '.']
Token IDs: [5050, 46035, 29736, 4641, 13]


In [13]:
# Load the pretrained model

import torch
from transformers import AutoModelForCausalLM

# Choose appropriate precision
dtype = (
    torch.float16
    if torch.cuda.is_available()
    else torch.float32
)

# Load pretrained model
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
    device_map="auto"
)

# Set evaluation mode
model.eval()

print("Model loaded successfully!")
print("Model device:", model.device)
print("Model parameters:",
      sum(p.numel() for p in model.parameters()))

print("GPU:", torch.cuda.get_device_name(0)
      if torch.cuda.is_available() else "CPU")


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Model device: cuda:0
Model parameters: 494032768
GPU: Tesla T4


In [15]:

prompt = "What should I do if my ATM card gets swallowed?"

messages = [
    {
        "role": "user",
        "content": prompt
    }
]

# Tokenize the conversation
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

# Move input tensors to the model's device
inputs = {
    key: value.to(model.device)
    for key, value in inputs.items()
}

# Check input shape
print("Input shape:", inputs["input_ids"].shape)

# Generate response
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs.get("attention_mask"),
        max_new_tokens=100,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

# Extract generated tokens only
new_tokens = outputs[0][
    inputs["input_ids"].shape[-1]:
]

# Convert tokens into readable text
response = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
)

print("Question:", prompt)
print("\nModel response:", response)


Input shape: torch.Size([1, 40])
Question: What should I do if my ATM card gets swallowed?

Model response: If your ATM card is swallowed, you can try the following steps to retrieve it:

1. **Check Your ATM Card**: Ensure that your ATM card is not damaged or missing any important information.

2. **Contact ATM Service**: If you have access to the ATM machine, contact the service desk for assistance. They may be able to help you locate and retrieve your card.

3. **Use a Different ATM**: Try using another ATM in the same location to see if the card returns.

4.


In [16]:

print("Tokenizer:", type(tokenizer).__name__)
print("Model:", type(model).__name__)
print("Model loaded:", model is not None)
print("Tokenizer loaded:", tokenizer is not None)

print(
    "Total parameters:",
    f"{sum(p.numel() for p in model.parameters()):,}"
)

print("GPU allocated memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2),
      "GB" if torch.cuda.is_available() else "N/A")


Tokenizer: Qwen2Tokenizer
Model: Qwen2ForCausalLM
Model loaded: True
Tokenizer loaded: True
Total parameters: 494,032,768
GPU allocated memory: 0.93 GB


In [17]:
# Step 4 - Baseline Evaluation

def create_prompt(text):
    return f"""You are a banking customer support intent classifier.

Classify the customer's message into exactly one of these intents:

{selected_intents}

Return only the intent name. Do not explain.

Customer message: {text}

Intent:"""

# Test prompt
print(create_prompt("My ATM withdrawal was declined."))


You are a banking customer support intent classifier.

Classify the customer's message into exactly one of these intents:

['cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'declined_cash_withdrawal', 'pending_cash_withdrawal', 'failed_transfer', 'card_swallowed']

Return only the intent name. Do not explain.

Customer message: My ATM withdrawal was declined.

Intent:


In [18]:
# Create a prediction function
# This function will:
# 1. Take a customer message.
# 2. Convert it into tokens.
# 3. Pass it to Qwen.
# 4. Generate a prediction.
# 5. Extract the predicted intent.
def predict_intent(model, tokenizer, text):
    prompt = create_prompt(text)

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            max_new_tokens=30,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[-1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Match the generated answer against valid categories
    prediction = response.lower().strip()

    if prediction in selected_intents:
        return prediction

    return "invalid_prediction"


In [19]:
# Test the prediction function

sample = test_data[0]

prediction = predict_intent(
    model,
    tokenizer,
    sample["text"]
)

actual = label_names[sample["label"]]

print("Customer message:", sample["text"])
print("Actual intent:", actual)
print("Predicted intent:", prediction)


Customer message: My withdrawl is still pending.  Why?
Actual intent: pending_cash_withdrawal
Predicted intent: pending_cash_withdrawal


In [20]:
# Evaluate the complete test dataset

from tqdm.auto import tqdm

y_true = []
y_pred = []

for example in tqdm(test_data):
    actual = label_names[example["label"]]

    prediction = predict_intent(
        model,
        tokenizer,
        example["text"]
    )

    y_true.append(actual)
    y_pred.append(prediction)

print("Evaluation completed!")
print("Total examples:", len(y_true))


  0%|          | 0/240 [00:00<?, ?it/s]

Evaluation completed!
Total examples: 240


In [21]:
# Calculate evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    labels=selected_intents,
    average="macro",
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    labels=selected_intents,
    average="macro",
    zero_division=0
)

macro_f1 = f1_score(
    y_true,
    y_pred,
    labels=selected_intents,
    average="macro",
    zero_division=0
)

print("BASE MODEL RESULTS")
print("------------------")
print("Accuracy:", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall:", round(recall, 4))
print("Macro F1:", round(macro_f1, 4))

print("\nDetailed Classification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=selected_intents,
        zero_division=0
    )
)


BASE MODEL RESULTS
------------------
Accuracy: 0.5417
Precision: 0.5774
Recall: 0.5417
Macro F1: 0.5265

Detailed Classification Report:
                                precision    recall  f1-score   support

        cash_withdrawal_charge       1.00      0.53      0.69        40
cash_withdrawal_not_recognised       0.47      0.53      0.49        40
      declined_cash_withdrawal       0.31      0.78      0.45        40
       pending_cash_withdrawal       0.93      0.65      0.76        40
               failed_transfer       0.76      0.78      0.77        40
                card_swallowed       0.00      0.00      0.00        40

                     micro avg       0.56      0.54      0.55       240
                     macro avg       0.58      0.54      0.53       240
                  weighted avg       0.58      0.54      0.53       240



In [22]:
# Save the baseline results
import pandas as pd
import os

os.makedirs("results", exist_ok=True)

baseline_df = pd.DataFrame({
    "text": [x["text"] for x in test_data],
    "actual_intent": y_true,
    "predicted_intent": y_pred
})

baseline_df.to_csv(
    "results/base_model_predictions.csv",
    index=False
)

baseline_metrics = pd.DataFrame([{
    "model": "Qwen2.5-0.5B-Instruct",
    "accuracy": accuracy,
    "precision_macro": precision,
    "recall_macro": recall,
    "f1_macro": macro_f1
}])

baseline_metrics.to_csv(
    "results/baseline_metrics.csv",
    index=False
)

print("Baseline results saved successfully!")


Baseline results saved successfully!


In [23]:
# Before applying LoRA, let's check the actual module names inside Qwen.

# Inspect attention projection modules
for name, module in model.named_modules():
    if any(
        target in name
        for target in [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj"
        ]
    ):
        print(name)


model.layers.0.self_attn.q_proj
model.layers.0.self_attn.k_proj
model.layers.0.self_attn.v_proj
model.layers.0.self_attn.o_proj
model.layers.1.self_attn.q_proj
model.layers.1.self_attn.k_proj
model.layers.1.self_attn.v_proj
model.layers.1.self_attn.o_proj
model.layers.2.self_attn.q_proj
model.layers.2.self_attn.k_proj
model.layers.2.self_attn.v_proj
model.layers.2.self_attn.o_proj
model.layers.3.self_attn.q_proj
model.layers.3.self_attn.k_proj
model.layers.3.self_attn.v_proj
model.layers.3.self_attn.o_proj
model.layers.4.self_attn.q_proj
model.layers.4.self_attn.k_proj
model.layers.4.self_attn.v_proj
model.layers.4.self_attn.o_proj
model.layers.5.self_attn.q_proj
model.layers.5.self_attn.k_proj
model.layers.5.self_attn.v_proj
model.layers.5.self_attn.o_proj
model.layers.6.self_attn.q_proj
model.layers.6.self_attn.k_proj
model.layers.6.self_attn.v_proj
model.layers.6.self_attn.o_proj
model.layers.7.self_attn.q_proj
model.layers.7.self_attn.k_proj
model.layers.7.self_attn.v_proj
model.la

In [24]:
# Create the LoRA configuration

from peft import LoraConfig, TaskType

# Define LoRA configuration
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

print("LoRA configuration created successfully!")
print(lora_config)


LoRA configuration created successfully!
LoraConfig(task_type=<TaskType.CAUSAL_LM: 'CAUSAL_LM'>, peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.21.2', base_model_name_or_path=None, revision=None, inference_mode=False, r=8, target_modules={'q_proj', 'k_proj', 'v_proj', 'o_proj'}, exclude_modules=None, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_config=None, kasa_conf

In [25]:
# Check the dataset structure

print("Training example:")
print(train_data[0])

print("\nValidation example:")
print(validation_data[0])

print("\nTraining columns:", train_data.column_names)
print("Validation columns:", validation_data.column_names)


Training example:
{'text': 'I hope you can help me. Someone has stolen my wallet and now I have noticed that they have been taking money from my account.', 'label': 20}

Validation example:
{'text': 'I was charged for a cash withdrawal, why?', 'label': 19}

Training columns: ['text', 'label']
Validation columns: ['text', 'label']


In [31]:

def preprocess_function(example):

    # Convert numeric label to intent name
    actual_label = label_names[example["label"]]

    user_message = create_prompt(example["text"])

    prompt_messages = [
        {
            "role": "user",
            "content": user_message
        }
    ]

    full_messages = [
        {
            "role": "user",
            "content": user_message
        },
        {
            "role": "assistant",
            "content": actual_label
        }
    ]

    # Tokenize prompt and extract input_ids
    prompt_encoding = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        truncation=True,
        max_length=512
    )

    prompt_ids = prompt_encoding["input_ids"]

    # Tokenize complete conversation
    full_encoding = tokenizer.apply_chat_template(
        full_messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        truncation=True,
        max_length=512
    )

    full_ids = full_encoding["input_ids"]

    # Make sure these are lists of token IDs
    if hasattr(prompt_ids, "tolist"):
        prompt_ids = prompt_ids.tolist()

    if hasattr(full_ids, "tolist"):
        full_ids = full_ids.tolist()

    prompt_length = len(prompt_ids)

    # Create labels from the complete conversation
    labels = full_ids.copy()

    # Ignore prompt tokens during loss calculation
    labels[:prompt_length] = [-100] * prompt_length

    return {
        "input_ids": full_ids,
        "attention_mask": [1] * len(full_ids),
        "labels": labels
    }


print("Corrected preprocessing function created!")


Corrected preprocessing function created!


In [32]:

train_tokenized = train_data.map(
    preprocess_function,
    remove_columns=train_data.column_names
)

validation_tokenized = validation_data.map(
    preprocess_function,
    remove_columns=validation_data.column_names
)

print("Tokenization completed successfully!")
print("Training examples:", len(train_tokenized))
print("Validation examples:", len(validation_tokenized))


Map:   0%|          | 0/765 [00:00<?, ? examples/s]

Map:   0%|          | 0/86 [00:00<?, ? examples/s]

Tokenization completed successfully!
Training examples: 765
Validation examples: 86


In [33]:

sample = train_tokenized[0]

print("Input IDs type:", type(sample["input_ids"]))
print("Input IDs:", sample["input_ids"][:10])

print("\nLabels type:", type(sample["labels"]))
print("Labels:", sample["labels"][:10])

print("\nFirst input token type:", type(sample["input_ids"][0]))
print("First label token type:", type(sample["labels"][0]))

# Extract answer tokens
answer_tokens = [
    token
    for token in sample["labels"]
    if token != -100
]

print("\nAnswer the model must learn:")
print(
    tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    )
)

print("\nIgnored prompt tokens:", sample["labels"].count(-100))


Input IDs type: <class 'list'>
Input IDs: [151644, 8948, 198, 2610, 525, 1207, 16948, 11, 3465, 553]

Labels type: <class 'list'>
Labels: [-100, -100, -100, -100, -100, -100, -100, -100, -100, -100]

First input token type: <class 'int'>
First label token type: <class 'int'>

Answer the model must learn:
cash_withdrawal_not_recognised


Ignored prompt tokens: 134


In [34]:

assert len(train_tokenized) == 765
assert len(validation_tokenized) == 86

assert isinstance(train_tokenized[0]["input_ids"], list)
assert isinstance(train_tokenized[0]["labels"], list)

assert all(
    isinstance(token, int)
    for token in train_tokenized[0]["input_ids"]
)

assert any(
    token != -100
    for token in train_tokenized[0]["labels"]
)

print("All corrected preprocessing checks passed!")


All corrected preprocessing checks passed!


In [36]:

import torch
from dataclasses import dataclass

@dataclass
class CausalLMCollator:

    tokenizer: object

    def __call__(self, features):

        # Pad input IDs and attention masks
        input_features = [
            {
                "input_ids": item["input_ids"],
                "attention_mask": item["attention_mask"]
            }
            for item in features
        ]

        batch = self.tokenizer.pad(
            input_features,
            padding=True,
            return_tensors="pt"
        )

        # Pad labels with -100
        max_length = batch["input_ids"].shape[1]

        padded_labels = []

        for item in features:
            labels = item["labels"]
            padding_length = max_length - len(labels)

            labels = labels + [-100] * padding_length
            padded_labels.append(labels)

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long
        )

        return batch


# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# Create collator
data_collator = CausalLMCollator(tokenizer)

print("Data collator created successfully!")


Data collator created successfully!


In [37]:

test_batch = data_collator([
    train_tokenized[0],
    train_tokenized[1]
])

print("Input batch shape:", test_batch["input_ids"].shape)
print("Attention mask shape:", test_batch["attention_mask"].shape)
print("Labels batch shape:", test_batch["labels"].shape)

print("\nAll dataset and batch checks passed!")


Input batch shape: torch.Size([2, 144])
Attention mask shape: torch.Size([2, 144])
Labels batch shape: torch.Size([2, 144])

All dataset and batch checks passed!


In [38]:
# Attach LoRA adapters and prepare training
# Check GPU and model

import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print("Model:", model.config._name_or_path)
print("Model dtype:", model.dtype)


GPU available: True
GPU name: Tesla T4
GPU memory: 14.56 GB
Model: Qwen/Qwen2.5-0.5B-Instruct
Model dtype: torch.float16


In [39]:
# Prepare the model for training

# Disable KV caching during training
model.config.use_cache = False

# Ensure the model has a padding token
model.config.pad_token_id = tokenizer.pad_token_id

print("Model prepared for LoRA training!")


Model prepared for LoRA training!


In [41]:

import torch
import peft
import transformers
import importlib.metadata as metadata

print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Transformers:", transformers.__version__)

try:
    print("TorchAO:", metadata.version("torchao"))
except metadata.PackageNotFoundError:
    print("TorchAO: Not installed")


PyTorch: 2.14.1+cu130
PEFT: 0.21.2
Transformers: 5.22.1
TorchAO: 0.10.0


In [42]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [44]:
# Attach LoRA
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

print("LoRA adapters attached successfully!")


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:154: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'Qwen/Qwen2.5-0.5B-Instruct' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184
LoRA adapters attached successfully!


In [45]:
# Reload a clean base model

import torch
import gc

from transformers import AutoModelForCausalLM

# Release the existing model
del model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Load a fresh original base model
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    ),
    device_map="auto"
)

print("Fresh base model loaded!")
print("Model type:", type(model))
print("Model device:", model.device)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Fresh base model loaded!
Model type: <class 'transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM'>
Model device: cuda:0


In [46]:
# Update the LoRA configuration

# Associate the configuration with the correct base model
lora_config.base_model_name_or_path = model_name

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

print("LoRA configuration ready!")


LoRA configuration ready!


In [47]:
# Attach LoRA exactly once

from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

print("\nLoRA adapters attached to fresh model!")


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184

LoRA adapters attached to fresh model!


In [48]:

from peft import PeftModel

assert isinstance(model, PeftModel)

trainable = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total = sum(
    p.numel()
    for p in model.parameters()
)

print("Trainable parameters:", trainable)
print("Total parameters:", total)
print("Trainable percentage:", round(trainable / total * 100, 4))

print("LoRA verification passed!")


Trainable parameters: 1081344
Total parameters: 495114112
Trainable percentage: 0.2184
LoRA verification passed!


In [50]:

import transformers
import inspect
from transformers import TrainingArguments

print("Transformers version:", transformers.__version__)

print("\nTrainingArguments location:")
print(inspect.getfile(TrainingArguments))

print("\nSupported arguments:")
print(inspect.signature(TrainingArguments.__init__))


Transformers version: 5.22.1

TrainingArguments location:
/usr/local/lib/python3.13/dist-packages/transformers/training_args.py

Supported arguments:
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 5e-05, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool = False, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None,

In [51]:
# TrainingArguments

import torch
import inspect

from transformers import TrainingArguments, Trainer

# Approximate total optimizer updates:
# 765 examples / batch size 4 / accumulation 4
# = 48 updates per epoch
# 48 updates x 3 epochs = 144 updates
# 5% of 144 is approximately 7 warmup steps

training_kwargs = {
    "output_dir": "./results/lora_training",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "warmup_steps": 7,
    "logging_steps": 10,
    "save_strategy": "epoch",
    "eval_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "save_total_limit": 2,
    "fp16": torch.cuda.is_available(),
    "bf16": False,
    "report_to": "none",
    "optim": "adamw_torch",
    "seed": 42
}

# Keep only arguments supported by the installed version
supported_args = inspect.signature(
    TrainingArguments.__init__
).parameters

training_kwargs = {
    key: value
    for key, value in training_kwargs.items()
    if key in supported_args
}

training_args = TrainingArguments(**training_kwargs)

print("Training configuration created!")
print("Transformers version:", __import__("transformers").__version__)
print("Warmup steps:", training_args.warmup_steps)
print("FP16 enabled:", training_args.fp16)
print("Effective batch size:", 16)


Training configuration created!
Transformers version: 5.22.1
Warmup steps: 7
FP16 enabled: True
Effective batch size: 16


In [52]:
# Create the Trainer

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": train_tokenized,
    "eval_dataset": validation_tokenized,
    "data_collator": data_collator
}

trainer_parameters = inspect.signature(
    Trainer.__init__
).parameters

if "processing_class" in trainer_parameters:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer created successfully!")


Trainer created successfully!


In [53]:
# Start training

print("Starting LoRA fine-tuning...")

train_result = trainer.train()

print("\nTraining completed!")
print("\nTraining metrics:")
print(train_result.metrics)


Starting LoRA fine-tuning...


Epoch,Training Loss,Validation Loss
1,0.058731,0.041999
2,0.032050,0.021182
3,0.018942,0.024390



Training completed!

Training metrics:
{'train_runtime': 160.5071, 'train_samples_per_second': 14.298, 'train_steps_per_second': 0.897, 'total_flos': 711948877947648.0, 'train_loss': 0.046358111780136824, 'epoch': 3.0}


In [54]:
# Evaluate and save

validation_results = trainer.evaluate()

print("Validation results:")
print(validation_results)

# Save the trained adapter
adapter_path = "./results/fine_tuned_lora_adapter"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("\nAdapter saved at:", adapter_path)


Training Loss,Validation Loss,Epoch
0.018942,0.021182,3


Validation results:
{'eval_loss': 0.021182497963309288}

Adapter saved at: ./results/fine_tuned_lora_adapter


In [56]:
# Prepare the fine-tuned model
# Use the best model selected by Trainer
model = trainer.model

# Enable caching for inference
model.config.use_cache = True

# Set evaluation mode
model.eval()

print("Fine-tuned model ready for evaluation!")

Fine-tuned model ready for evaluation!


In [57]:
# Evaluate all 240 test examples

from tqdm.auto import tqdm
import pandas as pd

y_true_finetuned = []
y_pred_finetuned = []

for example in tqdm(test_data):

    actual = label_names[example["label"]]

    prediction = predict_intent(
        model,
        tokenizer,
        example["text"]
    )

    y_true_finetuned.append(actual)
    y_pred_finetuned.append(prediction)

print("Fine-tuned evaluation completed!")
print("Total test examples:", len(y_true_finetuned))
print("Predictions:", len(y_pred_finetuned))


  0%|          | 0/240 [00:00<?, ?it/s]

Fine-tuned evaluation completed!
Total test examples: 240
Predictions: 240


In [58]:
# Calculate evaluation metrics

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

finetuned_accuracy = accuracy_score(
    y_true_finetuned,
    y_pred_finetuned
)

finetuned_precision = precision_score(
    y_true_finetuned,
    y_pred_finetuned,
    labels=selected_intents,
    average="macro",
    zero_division=0
)

finetuned_recall = recall_score(
    y_true_finetuned,
    y_pred_finetuned,
    labels=selected_intents,
    average="macro",
    zero_division=0
)

finetuned_f1 = f1_score(
    y_true_finetuned,
    y_pred_finetuned,
    labels=selected_intents,
    average="macro",
    zero_division=0
)

print("FINE-TUNED MODEL RESULTS")
print("------------------------")
print("Accuracy:", round(finetuned_accuracy, 4))
print("Precision:", round(finetuned_precision, 4))
print("Recall:", round(finetuned_recall, 4))
print("Macro F1:", round(finetuned_f1, 4))

print("\nDetailed Classification Report:")
print(
    classification_report(
        y_true_finetuned,
        y_pred_finetuned,
        labels=selected_intents,
        zero_division=0
    )
)


FINE-TUNED MODEL RESULTS
------------------------
Accuracy: 0.9458
Precision: 0.9468
Recall: 0.9458
Macro F1: 0.9455

Detailed Classification Report:
                                precision    recall  f1-score   support

        cash_withdrawal_charge       0.98      1.00      0.99        40
cash_withdrawal_not_recognised       0.97      0.88      0.92        40
      declined_cash_withdrawal       0.90      0.93      0.91        40
       pending_cash_withdrawal       0.93      0.95      0.94        40
               failed_transfer       0.93      1.00      0.96        40
                card_swallowed       0.97      0.93      0.95        40

                      accuracy                           0.95       240
                     macro avg       0.95      0.95      0.95       240
                  weighted avg       0.95      0.95      0.95       240



In [59]:
# Compare base model against fine-tuned model

comparison = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "Macro F1"
    ],
    "Base Model": [
        0.5417,
        0.5774,
        0.5417,
        0.5265
    ],
    "Fine-Tuned Model": [
        finetuned_accuracy,
        finetuned_precision,
        finetuned_recall,
        finetuned_f1
    ]
})

comparison["Improvement"] = (
    comparison["Fine-Tuned Model"]
    - comparison["Base Model"]
)

print(comparison.to_string(index=False))


   Metric  Base Model  Fine-Tuned Model  Improvement
 Accuracy      0.5417          0.945833     0.404133
Precision      0.5774          0.946836     0.369436
   Recall      0.5417          0.945833     0.404133
 Macro F1      0.5265          0.945522     0.419022


In [60]:
# Save the fine-tuned predictions and metrics

import os

os.makedirs("results", exist_ok=True)

# Save individual predictions
predictions_df = pd.DataFrame({
    "actual_intent": y_true_finetuned,
    "predicted_intent": y_pred_finetuned
})

predictions_df["correct"] = (
    predictions_df["actual_intent"]
    == predictions_df["predicted_intent"]
)

predictions_df.to_csv(
    "results/fine_tuned_predictions.csv",
    index=False
)

# Save evaluation metrics
comparison.to_csv(
    "results/model_comparison.csv",
    index=False
)

print("Prediction and comparison files saved!")


Prediction and comparison files saved!


In [61]:
# Identify three failure examples

# Find incorrectly classified examples
failure_cases = []

for i, example in enumerate(test_data):

    actual = y_true_finetuned[i]
    predicted = y_pred_finetuned[i]

    if actual != predicted:
        failure_cases.append({
            "customer_message": example["text"],
            "actual_intent": actual,
            "predicted_intent": predicted
        })

print("Total incorrect predictions:", len(failure_cases))

print("\nTHREE FAILURE EXAMPLES")
print("=" * 60)

for i, failure in enumerate(failure_cases[:3], start=1):

    print(f"\nFailure {i}")
    print("Customer message:", failure["customer_message"])
    print("Actual intent:", failure["actual_intent"])
    print("Predicted intent:", failure["predicted_intent"])


Total incorrect predictions: 13

THREE FAILURE EXAMPLES

Failure 1
Customer message: Where's accounting for my cash withdrawal?
Actual intent: pending_cash_withdrawal
Predicted intent: cash_withdrawal_not_recognised

Failure 2
Customer message: My account was charged for a withdraw I tried to make that was decline.
Actual intent: pending_cash_withdrawal
Predicted intent: declined_cash_withdrawal

Failure 3
Customer message: I want to know why I have been unable to withdrawn funds.
Actual intent: declined_cash_withdrawal
Predicted intent: failed_transfer


In [62]:

import pandas as pd
import os

os.makedirs("results", exist_ok=True)

failures_df = pd.DataFrame(failure_cases)

failures_df.to_csv(
    "results/failure_cases.csv",
    index=False
)

print("Failure cases saved successfully!")


Failure cases saved successfully!


In [63]:
# Save your failure analysis

failure_analysis = [
    {
        "failure": 1,
        "reason": "Ambiguous withdrawal accounting/status wording",
        "actual_intent": "pending_cash_withdrawal",
        "predicted_intent": "cash_withdrawal_not_recognised"
    },
    {
        "failure": 2,
        "reason": "Conflicting signals: declined withdrawal and account charged",
        "actual_intent": "pending_cash_withdrawal",
        "predicted_intent": "declined_cash_withdrawal"
    },
    {
        "failure": 3,
        "reason": "Ambiguous withdrawal failure confused with transfer failure",
        "actual_intent": "declined_cash_withdrawal",
        "predicted_intent": "failed_transfer"
    }
]

import json

with open("results/failure_analysis.json", "w") as f:
    json.dump(failure_analysis, f, indent=4)

print("Failure analysis saved!")


Failure analysis saved!


In [64]:

import shutil
from google.colab import files

# Zip only the results folder
shutil.make_archive(
    "/content/results",
    "zip",
    root_dir="/content",
    base_dir="results"
)

# Download
files.download("/content/results.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>